In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"

In [63]:
from concept_abstraction.training import train_ppo_model, SimpleQEstimator
from concept_abstraction.selection import greedy_selection_supervised, lp_selection_supervised, lp_selection_supervised_imperfect, multiple_selection_supervised, iterative_selection_supervised, greedy_selection_supervised
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import torch.nn as nn
from torchvision import models
import torch
from torchvision import transforms
from torch.utils.data import DataLoader


In [4]:
is_jupyter = 'ipykernel' in sys.modules

In [5]:
if is_jupyter: 
    seed        = 42
    num_concepts_selected = 112
    out_folder = "cub"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    num_concepts_selected = args.num_concepts_selected
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [6]:
results = {}
results['parameters'] = {'seed'      : seed,
        'num_concepts_selected': num_concepts_selected,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'num_concepts_selected': 112}


In [7]:
np.random.seed(seed)
random.seed(seed)

In [8]:
dataset = json.load(open("../../data/cub/preprocessed.json"))

In [9]:
def get_performance(selected_concepts,accuracy_by_concept):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
        max_iter=1000,  # increase if needed
        random_state=0,
        alpha=1e-3,  # instead of 0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
    )

    # Train the model
    mlp.fit(train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

## Perfrect Concepts

In [10]:
results['perfect'] = {}

In [86]:
train_X = np.load("../../data/cub/cub_train_x.npy")
test_X = np.load("../../data/cub/cub_test_x.npy")
train_Y = np.array([row['label'] for row in dataset['train']])
test_Y = np.array([row['label'] for row in dataset['test']])
val_X = test_X[:1000,:]
val_Y = test_Y[:1000]
test_X = test_X[1000:,:]
test_Y = test_Y[1000:]

In [13]:
results['perfect']['lp'] = {}
for num_concepts_selected in range(10,311,10):
    lp_concept_list = lp_selection_supervised(train_X,train_Y,num_concepts_selected)
    results['perfect']['lp'][num_concepts_selected] = {
        'reward': get_performance(lp_concept_list,np.ones(312)),
        'concepts': lp_concept_list
    }
    print("LP Performance {}: {}".format(num_concepts_selected,results['perfect']['lp'][num_concepts_selected] ))

Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
LP Performance 10: {'reward': 0.6128493950771798, 'concepts': [6, 10, 20, 51, 54, 163, 218, 253, 289, 308]}
LP Performance 20: {'reward': 0.9380475594493116, 'concepts': [4, 6, 10, 20, 21, 51, 54, 75, 101, 149, 187, 193, 218, 236, 244, 249, 274, 283, 289, 311]}
LP Performance 30: {'reward': 0.9814351272423864, 'concepts': [6, 10, 20, 29, 36, 45, 51, 64, 75, 118, 126, 132, 149, 178, 187, 193, 218, 220, 227, 236, 240, 244, 249, 262, 274, 289, 293, 298, 309, 311]}
LP Performance 40: {'reward': 0.975177304964539, 'concepts': [6, 29, 56, 64, 75, 90, 117, 125, 126, 132, 133, 134, 144, 145, 147, 149, 157, 166, 167, 183, 194, 208, 209, 210, 218, 220, 227, 228, 243, 244, 249, 268, 274, 277, 283, 284, 292, 299, 308, 309]}
LP Performance 50: {'reward': 0.9814351272423864, 'concepts': [6, 29, 56, 64, 75, 90, 91, 117, 120, 125, 126, 132, 133, 134, 144, 145, 147, 149, 1

In [14]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]
results['perfect']['manual'] = {
    'reward': get_performance(manually_selected_concepts,np.ones(312)),
    'concepts': manually_selected_concepts
}
print("Manual Performance {}".format(results['perfect']['manual']['reward']))

Manual Performance 0.9814351272423864


#### Imperfect Concepts

In [15]:
img_locations = ["../../data/cub/images/{}".format(i['location']) for i in dataset['train']]
img_locations_test = ["../../data/cub/images/{}".format(i['location']) for i in dataset['test']]
img_locations_val = img_locations_test[:1000]
img_locations_test = img_locations_test[1000:]

In [16]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class CUBArrayDataset(Dataset):
    def __init__(self, img_locations, attributes, transform=None):
        """
        img_locations : list of image file paths
        attributes    : numpy array or torch tensor of shape (N, 312)
        transform     : torchvision transforms
        """
        self.img_locations = img_locations
        self.attributes = torch.tensor(attributes, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.img_locations)

    def __getitem__(self, idx):
        img = Image.open(self.img_locations[idx]).convert("RGB")
        label = self.attributes[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


In [17]:
resol = 299 # Inception V3 requires 299x299
resized_resol = 299 # not needed for RandomResizedCrop

# Training transforms
train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=32/255, saturation=(0.5, 1.5)),  # Match paper's exact params
    transforms.RandomResizedCrop(resol),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet normalization
])

# Validation/test transforms (center crop + resize)
val_transform = transforms.Compose([
    transforms.Resize(resol),
    transforms.CenterCrop(resol),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [18]:
from torch.utils.data import DataLoader

train_dataset = CUBArrayDataset(img_locations, train_X, transform=train_transform)
val_dataset = CUBArrayDataset(img_locations_val, val_X, transform=val_transform)  # Add validation set

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)  # batch_size=64
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)


In [19]:
model = models.inception_v3(pretrained=True, aux_logits=True)  # Use Inception V3

# Replace final FC layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 312)  # No sigmoid here - apply in loss

# Also replace auxiliary classifier if using aux_logits
if model.aux_logits:
    num_aux_features = model.AuxLogits.fc.in_features
    model.AuxLogits.fc = nn.Linear(num_aux_features, 312)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [90]:
pos_weight = []

for i in range(train_X.shape[1]):
    pos_count = train_X[:, i].sum().item()
    neg_count = len(train_X) - pos_count
    
    if pos_count == 0:
        weight = 1.0
    else:
        weight = neg_count / pos_count
        # Cap at a reasonable maximum (paper says ~9 average, so cap at 3-4x that)
        weight = min(weight, 30.0)  # or 50.0 max
    
    pos_weight.append(weight)

pos_weight = torch.tensor(pos_weight).to(device)

print(f"Average pos_weight: {pos_weight.mean().item():.2f}")
print(f"Min: {pos_weight.min().item():.2f}, Max: {pos_weight.max().item():.2f}")

Average pos_weight: 18.51
Min: 0.20, Max: 30.00


In [20]:

# Use BCEWithLogitsLoss with pos_weight for class imbalance
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# SGD with momentum 0.9 (start with one LR, will do hyperparameter search)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0004)


In [21]:
threshold = 0.5
num_epochs = 50
best_val_acc = 0.0

for epoch in range(num_epochs):
    # ===== Training Phase =====
    model.train()
    running_loss = 0.0
    correct_attrs = 0
    total_attrs = 0
    
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Inception V3 returns (outputs, aux_outputs) during training
        if model.training and model.aux_logits:
            outputs, aux_outputs = model(imgs)
            loss1 = criterion(outputs, labels)
            loss2 = criterion(aux_outputs, labels)
            loss = loss1 + 0.4 * loss2  # Standard Inception V3 aux loss weighting
        else:
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * imgs.size(0)
        
        # Accuracy calculation (apply sigmoid to logits)
        preds = (torch.sigmoid(outputs) > threshold).float()
        correct_attrs += (preds == labels).sum().item()
        total_attrs += labels.numel()
    
    epoch_loss = running_loss / len(train_loader.dataset)
    train_attr_accuracy = correct_attrs / total_attrs
    
    # ===== Validation Phase =====
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = (torch.sigmoid(outputs) > threshold).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.numel()
    
    val_attr_accuracy = val_correct / val_total
    
    # Track best model
    if val_attr_accuracy > best_val_acc:
        best_val_acc = val_attr_accuracy
        torch.save(model.state_dict(), 'best_model.pth')
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f} "
          f"Train Acc: {train_attr_accuracy:.4f} "
          f"Val Acc: {val_attr_accuracy:.4f}")


Epoch [1/50] Train Loss: 1.2118 Train Acc: 0.7120 Val Acc: 0.8022
Epoch [2/50] Train Loss: 0.9289 Train Acc: 0.8163 Val Acc: 0.8806
Epoch [3/50] Train Loss: 0.7739 Train Acc: 0.8675 Val Acc: 0.9031
Epoch [4/50] Train Loss: 0.6768 Train Acc: 0.8884 Val Acc: 0.9143
Epoch [5/50] Train Loss: 0.6138 Train Acc: 0.9010 Val Acc: 0.9221
Epoch [6/50] Train Loss: 0.5712 Train Acc: 0.9093 Val Acc: 0.9252
Epoch [7/50] Train Loss: 0.5317 Train Acc: 0.9148 Val Acc: 0.9336
Epoch [8/50] Train Loss: 0.5019 Train Acc: 0.9195 Val Acc: 0.9371
Epoch [9/50] Train Loss: 0.4810 Train Acc: 0.9233 Val Acc: 0.9401
Epoch [10/50] Train Loss: 0.4667 Train Acc: 0.9265 Val Acc: 0.9422
Epoch [11/50] Train Loss: 0.4455 Train Acc: 0.9293 Val Acc: 0.9445
Epoch [12/50] Train Loss: 0.4343 Train Acc: 0.9310 Val Acc: 0.9468
Epoch [13/50] Train Loss: 0.4152 Train Acc: 0.9336 Val Acc: 0.9508
Epoch [14/50] Train Loss: 0.4043 Train Acc: 0.9351 Val Acc: 0.9538
Epoch [15/50] Train Loss: 0.3948 Train Acc: 0.9368 Val Acc: 0.9539
Epoc

In [32]:
from sklearn.metrics import f1_score

model.eval()
all_labels = []
all_preds  = []

with torch.no_grad():
    for imgs, labels in train_loader:  # or a separate val/test loader
        imgs  = imgs.to(device)
        labels = labels.cpu().numpy()           # (batch, 312)

        outputs = model(imgs).cpu().numpy()     # probabilities
        preds = (outputs > 0.5).astype(np.int32)  # threshold

        all_labels.append(labels)
        all_preds.append(preds)

# Stack all batches
all_labels = np.vstack(all_labels)  # shape (N, 312)
all_preds  = np.vstack(all_preds)   # shape (N, 312)

# F1 per attribute (macro across 312)
f1_macro = f1_score(all_labels, all_preds, average="macro")
print(f"Macro F1 (avg over 312 attributes): {f1_macro:.4f}")


Macro F1 (avg over 312 attributes): 0.3640


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [23]:
resol = 299  # Inception V3 uses 299x299, not 224

test_transform = transforms.Compose([
    transforms.Resize(resol),  # First resize the image
    transforms.CenterCrop(resol),  # Then center crop to 299x299
    transforms.ToTensor(),  # divides by 255
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

train_pred_dataset = CUBArrayDataset(
    img_locations,
    torch.zeros((len(img_locations), 312)),
    transform=test_transform
)
test_pred_dataset = CUBArrayDataset(
    img_locations_test,
    torch.zeros((len(img_locations_test), 312)),
    transform=test_transform
)
train_pred_loader = DataLoader(train_pred_dataset, batch_size=64, shuffle=False, num_workers=4)  # batch_size=64
test_pred_loader = DataLoader(test_pred_dataset, batch_size=64, shuffle=False, num_workers=4)  # batch_size=64


/tmp/ipykernel_1200679/150385818.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.attributes = torch.tensor(attributes, dtype=torch.float32)


In [24]:
def get_predictions(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)           # probabilities after sigmoid
            preds.append(outputs.cpu().numpy())
    return np.vstack(preds)                 # shape: (N, 312)


In [25]:
pred_train_X = get_predictions(model, train_pred_loader, device)
pred_test_X  = get_predictions(model, test_pred_loader,  device)

print("pred_train_X:", pred_train_X.shape)
print("pred_test_X :", pred_test_X.shape)


pred_train_X: (5994, 312)
pred_test_X : (4794, 312)


In [52]:
from sklearn.neural_network import MLPClassifier

def get_performance_real(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128,128),
        activation='relu',
        solver='adam',
        max_iter=1000,  # increase if needed
        random_state=0,
        alpha=1e-3,  # instead of 0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
    )

    # Train the model
    mlp.fit(pred_train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(pred_test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

In [68]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128),
    activation='relu',
    solver='adam',
    max_iter=1000,  # increase if needed
    random_state=0,
    alpha=1e-3,  # instead of 0.0001,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
)

# Train the model
mlp.fit(pred_train_X, train_Y)

# Predict on the test set
y_pred = mlp.predict(pred_test_X)

# Compute accuracy
acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
acc

0.5846891948268669

In [27]:
results['imperfect'] = {}

In [28]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]


In [53]:
results['imperfect']['manual'] = {'reward': get_performance_real(manually_selected_concepts), 'concepts': manually_selected_concepts}

In [30]:
results['imperfect']['lp'] = {}
for c in results['perfect']['lp']:
    results['imperfect']['lp'][c] = {
        'reward': get_performance_real(results['perfect']['lp'][c]['concepts']),
        'concepts': c
    }

In [34]:
results['imperfect']['multiple'] = {}
for c in results['perfect']['lp']:
    imperfect_concepts = multiple_selection_supervised(train_X,train_Y,c)
    results['imperfect']['multiple'][c] = {
        'reward': get_performance_real(imperfect_concepts), 
        'concepts': imperfect_concepts
    }
results['imperfect']['multiple']

{10: {'reward': 0.5619524405506884,
  'concepts': [6, 20, 51, 54, 151, 209, 218, 235, 244, 289]},
 20: {'reward': 0.5607008760951189,
  'concepts': [6,
   7,
   20,
   35,
   51,
   54,
   117,
   132,
   149,
   151,
   163,
   209,
   218,
   235,
   236,
   244,
   259,
   289,
   304,
   308]},
 30: {'reward': 0.5826032540675845,
  'concepts': [6,
   7,
   10,
   14,
   20,
   35,
   51,
   54,
   69,
   117,
   131,
   132,
   149,
   151,
   163,
   178,
   193,
   209,
   218,
   220,
   235,
   236,
   240,
   244,
   253,
   259,
   260,
   289,
   304,
   308]},
 40: {'reward': 0.5949103045473508,
  'concepts': [6,
   7,
   10,
   14,
   20,
   21,
   25,
   35,
   45,
   51,
   54,
   69,
   101,
   116,
   117,
   131,
   132,
   149,
   151,
   163,
   178,
   193,
   194,
   209,
   218,
   220,
   235,
   236,
   240,
   244,
   249,
   253,
   254,
   259,
   260,
   274,
   289,
   304,
   308,
   311]},
 50: {'reward': 0.5919899874843555,
  'concepts': [6,
   7,
   10

In [58]:
results['imperfect']['iterative'] = {}
all_imperfect_concepts = iterative_selection_supervised(pred_train_X,train_Y,310)

for c in results['perfect']['lp']:
    imperfect_concepts = all_imperfect_concepts[:c]
    results['imperfect']['iterative'][c] = {
        'reward': get_performance_real(imperfect_concepts), 
        'concepts': imperfect_concepts
    }
results['imperfect']['iterative']

{10: {'reward': 0.416979557780559,
  'concepts': [204, 205, 206, 207, 208, 209, 210, 211, 212, 214]},
 20: {'reward': 0.5467250730079266,
  'concepts': [204,
   205,
   206,
   207,
   208,
   209,
   210,
   211,
   212,
   214,
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9]},
 30: {'reward': 0.549645390070922,
  'concepts': [204,
   205,
   206,
   207,
   208,
   209,
   210,
   211,
   212,
   214,
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19]},
 40: {'reward': 0.571130579891531,
  'concepts': [204,
   205,
   206,
   207,
   208,
   209,
   210,
   211,
   212,
   214,
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29]},
 50: {'reward': 0.5876095118898623,
  'concepts': [204,
   205,
   206,
   207,
   208,
   209,
   210,
   211,
   212,


In [61]:
results['imperfect']['random'] = {}

for c in results['perfect']['lp']:
    random_concepts = random.sample(list(range(312)),c)
    results['imperfect']['random'][c] = {
        'reward': get_performance_real(random_concepts), 
        'concepts': random_concepts
    }
results['imperfect']['random']

{10: {'reward': 0.4787234042553192,
  'concepts': [296, 277, 31, 160, 29, 25, 299, 244, 257, 271]},
 20: {'reward': 0.5379641218189404,
  'concepts': [80,
   29,
   260,
   41,
   95,
   35,
   304,
   34,
   120,
   206,
   61,
   291,
   126,
   296,
   20,
   214,
   298,
   289,
   267,
   161]},
 30: {'reward': 0.5387984981226533,
  'concepts': [133,
   104,
   160,
   122,
   135,
   202,
   67,
   153,
   234,
   161,
   37,
   4,
   288,
   51,
   275,
   109,
   259,
   178,
   35,
   125,
   189,
   145,
   80,
   224,
   278,
   154,
   270,
   283,
   53,
   68]},
 40: {'reward': 0.5673758865248227,
  'concepts': [135,
   59,
   54,
   283,
   79,
   139,
   144,
   309,
   107,
   175,
   104,
   258,
   250,
   128,
   26,
   47,
   216,
   141,
   22,
   1,
   170,
   66,
   134,
   82,
   226,
   282,
   218,
   287,
   4,
   57,
   38,
   76,
   279,
   18,
   189,
   298,
   75,
   220,
   65,
   21]},
 50: {'reward': 0.5725907384230288,
  'concepts': [157,
   186,
  

In [65]:
results['imperfect']['greedy'] = {}

for c in results['perfect']['lp']:
    greedy_concepts = greedy_selection_supervised(train_X,train_Y,c)
    results['imperfect']['greedy'][c] = {
        'reward': get_performance_real(greedy_concepts), 
        'concepts': greedy_concepts
    }
results['imperfect']['greedy']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


{10: {'reward': 0.492282019190655,
  'concepts': array([289,  54, 235,  51, 151,   6, 209,  20, 244, 218])},
 20: {'reward': 0.5640383813099707,
  'concepts': array([289,  54, 235,  51, 151,   6, 209,  20, 244, 218, 149, 117, 132,
          35, 304, 163, 236, 259,   7, 308])},
 30: {'reward': 0.5767626199415936,
  'concepts': array([289,  54, 235,  51, 151,   6, 209,  20, 244, 218, 149, 117, 132,
          35, 304, 163, 236, 259,   7, 308,  69, 193,  10, 178, 240, 260,
         220, 131,  14, 253])},
 40: {'reward': 0.5767626199415936,
  'concepts': array([289,  54, 235,  51, 151,   6, 209,  20, 244, 218, 149, 117, 132,
          35, 304, 163, 236, 259,   7, 308,  69, 193,  10, 178, 260, 240,
         220, 131,  14, 253,  25,  21, 101, 116, 249, 274, 194, 311, 254,
          45])},
 50: {'reward': 0.580517313308302,
  'concepts': array([289,  54, 235,  51, 151,   6, 209,  20, 244, 218, 149, 117, 132,
          35, 304, 163, 236, 259,   7, 308,  69, 193,  10, 178, 240, 260,
         220

## Save Data

In [66]:
save_path = get_save_path(out_folder,save_name)

In [67]:
delete_duplicate_results(out_folder,"",results)

In [68]:
json.dump(results,open('../../results/'+save_path,'w'))